# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print(f"Dataset: {metadata.get('name', '')}\n\n{metadata.get('description', '')}\n")
print(f"Published: {metadata.get('datePublished', '')}\nIdentifier: {metadata.get('identifier', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.


In [ ]:
# List all available record sets, fields, and columns
print("Available record sets (by @id):")
for rs in dataset.metadata.get('recordSet', []):
    if isinstance(rs, dict):
        print(f"  - {rs.get('@id')}")
    else:
        print(f"  - {rs}")

# For demonstration, let's show fields/columns of each record set
for rs in dataset.metadata.get('recordSet', []):
    print(f"\nRecord set: {rs.get('@id') if isinstance(rs, dict) else rs}")
    # Show fields
    if isinstance(rs, dict):
        fields = rs.get('field', [])
    else:
        rs_obj = next((r for r in dataset.metadata['recordSet'] if (isinstance(r, dict) and r.get('@id') == rs)), None)
        fields = rs_obj.get('field', []) if rs_obj else []
    print("Fields (by @id):")
    for f in fields:
        if isinstance(f, dict):
            print(f"  - {f.get('@id')}")
        else:
            print(f"  - {f}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.


In [ ]:
# Identify all record set @id(s)
record_sets = [
    rs['@id'] if isinstance(rs, dict) else rs
    for rs in dataset.metadata.get('recordSet', [])
]

dataframes = {}
# Only attempt to load records for record sets that have data
for record_set in record_sets:
    print(f"Loading records for record set: {record_set}")
    try:
        records = list(dataset.records(record_set=record_set))
        if len(records):
            df = pd.DataFrame(records)
            dataframes[record_set] = df
            print(f"  Columns: {df.columns.tolist()}")
            print(df.head())
        else:
            print("  [No records found]")
    except Exception as e:
        print(f"  [Could not load records: {e}]")

# For demonstration, pick the first record set if available
if len(record_sets):
    main_record_set_id = record_sets[0]
    if main_record_set_id in dataframes:
        print(f"\nColumns in {main_record_set_id}:\n", dataframes[main_record_set_id].columns.tolist())
        dataframes[main_record_set_id].head()
    else:
        print(f"No DataFrame extracted for {main_record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.


In [ ]:
# Select a record set with data
if len(dataframes):
    record_set_id = list(dataframes.keys())[0]  # Use the first one for demonstration
    df = dataframes[record_set_id]
    print(f"Using record set: {record_set_id}")
    
    # Try to find a numeric field
    import numpy as np
    numeric_field = None
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field = col
            break
    if not numeric_field:
        # Fallback: try to convert something
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                if np.issubdtype(df[col].dtype, np.number):
                    numeric_field = col
                    break
            except Exception:
                continue
    
    if numeric_field:
        print(f"Selected numeric field: {numeric_field}")
        threshold = df[numeric_field].quantile(0.75)  # Use 75th percentile as a dynamic threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())
        
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        
        # Try to pick a group field (categorical with low cardinality)
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].nunique() < min(10, len(df)//2):
                group_field = col
                break
        if group_field:
            print(f"\nGrouping by {group_field}:")
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric field found for this record set.")
else:
    print("No dataframes could be loaded in step 3.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) and 'numeric_field' in locals() and numeric_field:
    plt.figure(figsize=(7, 4))
    sns.histplot(dataframes[record_set_id][numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    
    # If filtering and normalization was successful, plot those too
    if 'filtered_df' in locals() and f"{numeric_field}_normalized" in filtered_df.columns:
        plt.figure(figsize=(7, 4))
        sns.scatterplot(
            data=filtered_df,
            x=numeric_field,
            y=f"{numeric_field}_normalized",
        )
        plt.title(f"{numeric_field} vs Normalized")
        plt.show()
else:
    print("No numeric field or data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.


- Loaded the Croissant metadata and identified available record sets.
- Extracted available record set(s) and performed basic data exploration using numeric fields for demonstration.
- Filtered high-value records, normalized a numeric feature, and visualized the distribution of a selected field.
- This notebook provides a flexible workflow for exploring any Croissant-compatible dataset with rich, schema-driven metadata.
